# Neural Network with Dropout Uncertainty for Classification

This notebook defines the neural network class for dropout uncertainty classification.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import math

try:
    from scipy.misc import logsumexp
except ImportError:
    from scipy.special import logsumexp
import numpy as np

from sklearn.neural_network import MLPClassifier

import time

In [2]:
class net:
    """
    Neural network class for Bayesian uncertainty estimation in classification tasks.
    """
    
    def __init__(self, X_train, y_train, n_hidden, n_classes=2, n_epochs=40,
        normalize=True, tau=1.0, dropout=0.05):
        """
            Constructor for the class implementing a Bayesian neural network
            trained with the probabilistic back propagation method for classification.

            @param X_train      Matrix with the features for the training data.
            @param y_train      Vector with the target variables for the
                                training data (class labels).
            @param n_hidden     Vector with the number of neurons for each
                                hidden layer.
            @param n_classes    Number of classes for classification.
            @param n_epochs     Number of epochs for which to train the
                                network.
            @param normalize    Whether to normalize the input features. This
                                is recommended unless the input vector is for
                                example formed by binary features (a
                                fingerprint). In that case we do not recommend
                                to normalize the features.
            @param tau          Tau value used for regularization
            @param dropout      Dropout rate for MC dropout simulation.
        """
        # Store parameters
        self.n_classes = n_classes
        self.tau = tau
        self.dropout = dropout
        
        # Normalize the training data if needed
        if normalize:
            self.std_X_train = np.std(X_train, 0)
            self.std_X_train[self.std_X_train == 0] = 1
            self.mean_X_train = np.mean(X_train, 0)
        else:
            self.std_X_train = np.ones(X_train.shape[1])
            self.mean_X_train = np.zeros(X_train.shape[1])

        X_train_norm = (X_train - np.full(X_train.shape, self.mean_X_train)) / \
                       np.full(X_train.shape, self.std_X_train)
        
        # Create a neural network classifier
        # For multiple hidden layers, create a tuple with layer sizes
        if isinstance(n_hidden, list):
            hidden_layer_sizes = tuple(n_hidden)
        else:
            hidden_layer_sizes = (n_hidden,)
        
        # Create model
        start_time = time.time()
        self.model = MLPClassifier(
            hidden_layer_sizes=hidden_layer_sizes,
            max_iter=n_epochs,
            alpha=1e-4,  # Regularization parameter
            solver='adam',
            activation='relu',
            learning_rate_init=0.001,
            random_state=1
        )
        
        # Train the model
        self.model.fit(X_train_norm, y_train)
        self.running_time = time.time() - start_time
        
    def predict(self, X_test, y_test):
        """
            Function for making predictions with the Bayesian neural network.

            @param X_test   The matrix of features for the test data
            @param y_test   The vector of target labels
            
            @return accuracy        The standard accuracy for the test data
            @return MC_accuracy     The Monte Carlo dropout accuracy for the test data
            @return test_ll         The test log-likelihood
        """
        X_test = np.array(X_test, ndmin=2)
        y_test = np.array(y_test, ndmin=1)

        # Normalize the test set
        X_test_norm = (X_test - np.full(X_test.shape, self.mean_X_train)) / \
                      np.full(X_test.shape, self.std_X_train)

        # Standard prediction (without dropout)
        y_prob = self.model.predict_proba(X_test_norm)
        y_pred = np.argmax(y_prob, axis=1)
        accuracy = np.mean(y_pred == y_test)

        # Monte Carlo dropout simulation
        T = 100  # Number of MC samples
        MC_predictions = []
        
        for _ in range(T):
            # Apply dropout to input features to simulate network dropout
            dropout_mask = np.random.binomial(1, 1-self.dropout, X_test_norm.shape)
            X_test_dropout = X_test_norm * dropout_mask
            
            # Get prediction
            probs = self.model.predict_proba(X_test_dropout)
            MC_predictions.append(probs)
        
        # Average predictions
        MC_pred_mean = np.mean(np.array(MC_predictions), axis=0)
        MC_pred_classes = np.argmax(MC_pred_mean, axis=1)
        MC_accuracy = np.mean(MC_pred_classes == y_test)

        # Compute test log-likelihood
        ll = 0
        for i, y in enumerate(y_test):
            ll += np.log(MC_pred_mean[i, int(y)] + 1e-10)  # Add small epsilon to avoid log(0)
        test_ll = ll / len(y_test)

        # We are done!
        return accuracy, MC_accuracy, test_ll

## Test the Neural Network

Let's test our neural network on a small toy dataset to ensure it's working correctly.

In [3]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# Create a toy dataset
X, y = make_classification(n_samples=1000, n_features=20, n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create and train a network
nn = net(X_train, y_train, n_hidden=[50], n_classes=2, n_epochs=20, normalize=True, tau=1.0, dropout=0.1)

# Make predictions
accuracy, MC_accuracy, test_ll = nn.predict(X_test, y_test)

print(f"Standard Accuracy: {accuracy:.4f}")
print(f"MC Dropout Accuracy: {MC_accuracy:.4f}")
print(f"Log-likelihood: {test_ll:.4f}")

Standard Accuracy: 0.8400
MC Dropout Accuracy: 0.8450
Log-likelihood: -0.4492
